In [1]:
"""
Property Graph Example using Neo4j

This script accompanies Section 3 of the chapter on Property Graphs.

Requirements
------------
pip install neo4j tabulate

Update the URI, username, and password below before running.
"""

'\nProperty Graph Example using Neo4j\n\nThis script accompanies Section 3 of the chapter on Property Graphs.\n\nRequirements\n------------\npip install neo4j tabulate\n\nUpdate the URI, username, and password below before running.\n'

In [2]:
from neo4j import GraphDatabase
from tabulate import tabulate

----------------------------------------------------------------------
Connection information
----------------------------------------------------------------------

In [3]:
URI = "bolt://localhost:7687"
AUTH = ("<userid>", "<password>")
driver = GraphDatabase.driver(
    URI,
    auth=AUTH
)

----------------------------------------------------------------------
Utility routine
----------------------------------------------------------------------

In [4]:
def run_query(query):

    print("=" * 72)
    print("neo4j>")
    print(query.strip())
    print()

    with driver.session() as session:
        result = session.run(query)

        records = list(result)
        columns = result.keys()

        if columns:
            rows = [[record[c] for c in columns] for record in records]
            print(tabulate(rows,
                           headers=columns,
                           tablefmt="github"))
        else:
            summary = result.consume()
            c = summary.counters

            print("Summary")
            print("-------")

            if c.nodes_created:
                print(f"Nodes created: {c.nodes_created}")

            if c.relationships_created:
                print(f"Relationships created: {c.relationships_created}")

            if c.properties_set:
                print(f"Properties set: {c.properties_set}")

        print()

----------------------------------------------------------------------
Clean database
----------------------------------------------------------------------

In [5]:
run_query("""
MATCH (n)
DETACH DELETE n
""")

neo4j>
MATCH (n)
DETACH DELETE n

Summary
-------



----------------------------------------------------------------------
Create nodes
We are using the design in Figure 3    
----------------------------------------------------------------------

In [6]:
run_query("""
CREATE
    (art:Person {name:'art'}),
    (bob:Person {name:'bob'}),
    (bea:Person {name:'bea', age:23}),
    (cal:Person {name:'cal'}),
    (cam:Person {name:'cam'}),
    (coe:Person {name:'coe'}),
    (cory:Person {name:'cory'}),
    (seattle:City {name:'seattle'})
""")

neo4j>
CREATE
    (art:Person {name:'art'}),
    (bob:Person {name:'bob'}),
    (bea:Person {name:'bea', age:23}),
    (cal:Person {name:'cal'}),
    (cam:Person {name:'cam'}),
    (coe:Person {name:'coe'}),
    (cory:Person {name:'cory'}),
    (seattle:City {name:'seattle'})

Summary
-------
Nodes created: 8
Properties set: 9



----------------------------------------------------------------------
Create relationships
----------------------------------------------------------------------

In [7]:
run_query("""
MATCH
    (art:Person {name:'art'}),
    (bob:Person {name:'bob'}),
    (bea:Person {name:'bea'}),
    (cal:Person {name:'cal'}),
    (cam:Person {name:'cam'}),
    (coe:Person {name:'coe'}),
    (cory:Person {name:'cory'}),
    (seattle:City {name:'seattle'})
CREATE
    (art)-[:KNOWS {since:2005}]->(bob),
    (art)-[:KNOWS {since:2012}]->(bea),
    (bob)-[:KNOWS]->(cal),
    (bob)-[:KNOWS]->(cam),
    (bea)-[:KNOWS]->(coe),
    (bea)-[:KNOWS]->(cory),
    (bea)-[:BASED_NEAR]->(seattle)
""")

neo4j>
MATCH
    (art:Person {name:'art'}),
    (bob:Person {name:'bob'}),
    (bea:Person {name:'bea'}),
    (cal:Person {name:'cal'}),
    (cam:Person {name:'cam'}),
    (coe:Person {name:'coe'}),
    (cory:Person {name:'cory'}),
    (seattle:City {name:'seattle'})
CREATE
    (art)-[:KNOWS {since:2005}]->(bob),
    (art)-[:KNOWS {since:2012}]->(bea),
    (bob)-[:KNOWS]->(cal),
    (bob)-[:KNOWS]->(cam),
    (bea)-[:KNOWS]->(coe),
    (bea)-[:KNOWS]->(cory),
    (bea)-[:BASED_NEAR]->(seattle)

Summary
-------
Relationships created: 7
Properties set: 2



----------------------------------------------------------------------
Query 1
----------------------------------------------------------------------

In [8]:
run_query("""
MATCH (p1:Person {name:'art'})-[:KNOWS]->(p2:Person)
RETURN p2.name AS Friend
""")

neo4j>
MATCH (p1:Person {name:'art'})-[:KNOWS]->(p2:Person)
RETURN p2.name AS Friend

| Friend   |
|----------|
| bob      |
| bea      |



----------------------------------------------------------------------
Query 2
----------------------------------------------------------------------

In [9]:
run_query("""
MATCH (p1:Person {name:'art'})-[r:KNOWS {since:2010}]->(p2:Person)
RETURN p2.name AS Friend
""")

neo4j>
MATCH (p1:Person {name:'art'})-[r:KNOWS {since:2010}]->(p2:Person)
RETURN p2.name AS Friend

| Friend   |
|----------|



----------------------------------------------------------------------
Query 3
----------------------------------------------------------------------

In [10]:
run_query("""
MATCH (p1:Person {name:'art'})-[r:KNOWS]->(p2:Person)
WHERE r.since <= 2010
RETURN p2.name AS Friend
""")

neo4j>
MATCH (p1:Person {name:'art'})-[r:KNOWS]->(p2:Person)
WHERE r.since <= 2010
RETURN p2.name AS Friend

| Friend   |
|----------|
| bob      |



----------------------------------------------------------------------
Query 4
----------------------------------------------------------------------

In [11]:
run_query("""
MATCH (p:Person {name:'bea'})-[:BASED_NEAR]->(c:City)
RETURN c.name AS City
""")

neo4j>
MATCH (p:Person {name:'bea'})-[:BASED_NEAR]->(c:City)
RETURN c.name AS City

| City    |
|---------|
| seattle |



In [12]:
driver.close()